In [1]:
from __future__ import division
from __future__ import absolute_import

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torchvision.datasets as dset
import torchvision.transforms as transforms
import models

device=torch.device("cuda:4")

/mnt/data/anaconda3/envs/pytorch_2_0/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/mnt/data/anaconda3/envs/pytorch_2_0/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
cleannet = models.__dict__['resnet56'](10).to(device)

CifarResNet : Depth : 56 , Layers for each block : 9


In [3]:
from thop import profile as thop_profile

inp = torch.randn(1, 3, 32, 32).to(device)
flops, params,redet = thop_profile(cleannet, inputs=(inp, ),verbose=True,ret_layer_info=True)

print("%s | %.2f | %.2f" % ('origin', params / (1000 ** 2), flops / (1000**2)))#这里除以1000的平方，是为了化成M的单位

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_avgpool() for <class 'torch.nn.modules.pooling.AvgPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
origin | 0.85 | 127.62


In [4]:
net = models.__dict__['cgc_resnet56'](10).to(device)

CifarResNet : Depth : 56 , Layers for each block : 9


In [5]:
data_path='/mnt/data/zms/ICCV25/Train_models/data'

In [6]:
check_point_path="/mnt/data/zms/ICCV25/CGConv/save/cifar10_cgc_resnet56_500_SGD_resnet56_cgc_333/model_best.pth.tar"

In [ ]:
checkpoint = torch.load(check_point_path,weights_only=False)
model_dict = net.state_dict()
pretrained_dict = {k:v for k, v in checkpoint['state_dict'].items() if k in model_dict}
model_dict.update(pretrained_dict)
net.load_state_dict(model_dict)

In [8]:
mean = [x / 255 for x in [125.3, 123.0, 113.9]]
std = [x / 255 for x in [63.0, 62.1, 66.7]]

In [9]:
train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])
test_transform = transforms.Compose(
    [transforms.ToTensor(),
        transforms.Normalize(mean, std)])

In [10]:
train_data = dset.CIFAR10(data_path,
                            train=True,
                            transform=train_transform,
                            download=True)
test_data = dset.CIFAR10(data_path,
                            train=False,
                            transform=test_transform,
                            download=True)

Files already downloaded and verified
Files already downloaded and verified


In [11]:
train_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_data,
                                            batch_size=128,
                                            shuffle=True,
                                            num_workers=4,
                                            pin_memory=True)

In [12]:
from cgToolkit import *

In [14]:
for name, layer in net.named_modules():
    if "conv" in name or "classifier" in name:
        print("\"",name,"\",",sep="")

"conv_1_3x3",
"stage_1.0.conv_a",
"stage_1.0.conv_b",
"stage_1.1.conv_a",
"stage_1.1.conv_b",
"stage_1.2.conv_a",
"stage_1.2.conv_b",
"stage_1.3.conv_a",
"stage_1.3.conv_b",
"stage_1.4.conv_a",
"stage_1.4.conv_b",
"stage_1.5.conv_a",
"stage_1.5.conv_b",
"stage_1.6.conv_a",
"stage_1.6.conv_b",
"stage_1.7.conv_a",
"stage_1.7.conv_b",
"stage_1.8.conv_a",
"stage_1.8.conv_b",
"stage_2.0.conv_a",
"stage_2.0.conv_b",
"stage_2.1.conv_a",
"stage_2.1.conv_b",
"stage_2.2.conv_a",
"stage_2.2.conv_b",
"stage_2.3.conv_a",
"stage_2.3.conv_b",
"stage_2.4.conv_a",
"stage_2.4.conv_b",
"stage_2.5.conv_a",
"stage_2.5.conv_b",
"stage_2.6.conv_a",
"stage_2.6.conv_b",
"stage_2.7.conv_a",
"stage_2.7.conv_b",
"stage_2.8.conv_a",
"stage_2.8.conv_b",
"stage_3.0.conv_a",
"stage_3.0.conv_b",
"stage_3.1.conv_a",
"stage_3.1.conv_b",
"stage_3.2.conv_a",
"stage_3.2.conv_b",
"stage_3.3.conv_a",
"stage_3.3.conv_b",
"stage_3.4.conv_a",
"stage_3.4.conv_b",
"stage_3.5.conv_a",
"stage_3.5.conv_b",
"stage_3.6.conv_a",
"stage

In [16]:
layer_names=[
"conv_1_3x3",
"stage_1.0.conv_a",
"stage_1.0.conv_b",
"stage_1.1.conv_a",
"stage_1.1.conv_b",
"stage_1.2.conv_a",
"stage_1.2.conv_b",
"stage_1.3.conv_a",
"stage_1.3.conv_b",
"stage_1.4.conv_a",
"stage_1.4.conv_b",
"stage_1.5.conv_a",
"stage_1.5.conv_b",
"stage_1.6.conv_a",
"stage_1.6.conv_b",
"stage_1.7.conv_a",
"stage_1.7.conv_b",
"stage_1.8.conv_a",
"stage_1.8.conv_b",
"stage_2.0.conv_a",
"stage_2.0.conv_b",
"stage_2.1.conv_a",
"stage_2.1.conv_b",
"stage_2.2.conv_a",
"stage_2.2.conv_b",
"stage_2.3.conv_a",
"stage_2.3.conv_b",
"stage_2.4.conv_a",
"stage_2.4.conv_b",
"stage_2.5.conv_a",
"stage_2.5.conv_b",
"stage_2.6.conv_a",
"stage_2.6.conv_b",
"stage_2.7.conv_a",
"stage_2.7.conv_b",
"stage_2.8.conv_a",
"stage_2.8.conv_b",
"stage_3.0.conv_a",
"stage_3.0.conv_b",
"stage_3.1.conv_a",
"stage_3.1.conv_b",
"stage_3.2.conv_a",
"stage_3.2.conv_b",
"stage_3.3.conv_a",
"stage_3.3.conv_b",
"stage_3.4.conv_a",
"stage_3.4.conv_b",
"stage_3.5.conv_a",
"stage_3.5.conv_b",
"stage_3.6.conv_a",
"stage_3.6.conv_b",
"stage_3.7.conv_a",
"stage_3.7.conv_b",
"stage_3.8.conv_a",
"stage_3.8.conv_b",
# "classifier",
]
len(layer_names)

55

In [19]:
def inference_fun(model,dataloader=test_loader):
    with torch.no_grad():
        for i, (inputs, labels) in enumerate(dataloader):
            inputs = inputs.to(device)
            outputs = model(inputs)

In [21]:
states1=flops_ratio(model=net,inference_func=inference_fun,layer_list=layer_names,f_type='Sponge')

55


In [22]:
# states1.k1_s1_fired_perc
k1_s1_fired_perc=list(states1.k1_s1_fired_perc.values())
k3_s1_fired_perc=list(states1.k3_s1_fired_perc.values())
k3_s2_fired_perc=list(states1.k3_s2_fired_perc.values())

In [23]:
states0=flops_ratio(model=net,inference_func=inference_fun,layer_list=layer_names,f_type='Sponge0')

55


In [24]:
# states1.k1_s1_fired_perc
k1_s1_fired_perc_f2l=list(states0.k1_s1_fired_perc.values())
k3_s1_fired_perc_f2l=list(states0.k3_s1_fired_perc.values())
k3_s2_fired_perc_f2l=list(states0.k3_s2_fired_perc.values())

In [25]:
def flatten_structure(data, parent_key='', sep='.', result=None):
    if result is None:
        result = {}
    for key, value in data.items():
        new_key = f"{parent_key}{sep}{key}" if parent_key else key
        # 提取当前层的数据，第三个元素置为空字典
        result[new_key] = (value[0], value[1], {})
        # 递归处理子层
        if isinstance(value[2], dict) and value[2]:
            flatten_structure(value[2], new_key, sep, result)
    return result

In [26]:
flatten_redet=flatten_structure(redet)

In [27]:
flops_list=[flatten_redet[i][0] for i in layer_names]
classifier_flops=flatten_redet["classifier"][0]

In [28]:
len(flops_list)

55

In [31]:
list_cgc=[
0,
0,0,0,0, 0,0,1,1,1,1,1,1,1,1, 1,1,1,1,
0,0,0,0, 0,0,1,1,1,1,1,1,1,1, 1,1,1,1,
0,0,0,0, 0,0,1,1,1,1,1,1,1,1, 1,1,1,1,
]
len(list_cgc)

55

In [ ]:
flops_cgc = classifier_flops

for i in range(0,len(flops_list)):
    if list_cgc[i]:
        flops_cgc+=flops_list[i]*k3_s1_fired_perc[i]
    else:
        flops_cgc+=flops_list[i]*k3_s1_fired_perc_f2l[i]

print(flops_cgc)
print(sum(flops_list))
print(1-flops_cgc/(sum(flops_list)))